
## Folder 04 — File 06: Silver Shipments:


**Transformation rules:**

- Trim shipment, order, carrier, and tracking identifiers.
- Standardize carrier and shipment status values.
- Convert shipped_date and delivered_date into timestamps.
- Validate shipment status against allowed values.
- Check delivery-date consistency.
- Quarantine invalid records.
- Deduplicate using shipment_id.

In [0]:

# ============================================================
# SILVER SHIPMENTS - COMPLETE UPDATED NOTEBOOK
# ============================================================


# 1. IMPORT LIBRARIES

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Libraries imported successfully.")


# 2. PROJECT CONFIGURATION

CATALOG = "workspace"

BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"

BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_shipments"
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_shipments"
QUARANTINE_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.quarantine_shipments"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

print("Silver schema is ready.")
print("Bronze table:", BRONZE_TABLE)
print("Silver table:", SILVER_TABLE)
print("Quarantine table:", QUARANTINE_TABLE)


# 3. READ BRONZE SHIPMENT DATA

bronze_df = spark.table(BRONZE_TABLE)

bronze_count = bronze_df.count()

print("BRONZE SHIPMENT RECORDS:", bronze_count)

display(bronze_df.limit(10))


# 4. VALIDATE SOURCE COLUMNS

required_columns = [
    "shipment_id",
    "order_id",
    "carrier",
    "shipment_status",
    "tracking_number",
    "shipped_date",
    "delivered_date"
]

missing_columns = [
    column
    for column in required_columns
    if column not in bronze_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required source columns: {missing_columns}"
    )

print("Source column validation passed.")


# 5. STANDARDIZE SHIPMENT DATA

# Remove any previously generated validation column.
# This prevents old quarantine reasons from being reused.

if "_quarantine_reason" in bronze_df.columns:
    bronze_df = bronze_df.drop("_quarantine_reason")

standardized_df = (
    bronze_df

    .withColumn(
        "shipment_id",
        F.trim(F.col("shipment_id").cast("string"))
    )

    .withColumn(
        "order_id",
        F.trim(F.col("order_id").cast("string"))
    )

    .withColumn(
        "carrier",
        F.upper(F.trim(F.col("carrier").cast("string")))
    )

    .withColumn(
        "shipment_status",
        F.upper(F.trim(F.col("shipment_status").cast("string")))
    )

    .withColumn(
        "tracking_number",
        F.trim(F.col("tracking_number").cast("string"))
    )

    .withColumn(
        "shipped_date",
        F.to_timestamp("shipped_date")
    )

    .withColumn(
        "delivered_date",
        F.to_timestamp("delivered_date")
    )
)

print("Shipment standardization completed.")

display(standardized_df.limit(10))


# 6. DEFINE ALLOWED SHIPMENT STATUSES

ALLOWED_STATUSES = [
    "PENDING",
    "PROCESSING",
    "SHIPPED",
    "IN_TRANSIT",
    "OUT_FOR_DELIVERY",
    "DELIVERED",
    "CANCELLED",
    "RETURNED",
    "RETURN_TO_ORIGIN"
]

print("Allowed shipment statuses:")
print(ALLOWED_STATUSES)


# 7. DATA QUALITY VALIDATION

# Business rules:
#
# 1. shipment_id cannot be null or empty.
# 2. order_id cannot be null or empty.
# 3. shipment_status must be an allowed status.
# 4. shipped_date is mandatory.
# 5. DELIVERED shipments must have delivered_date.
# 6. Non-delivered shipments cannot have delivered_date,
#    except RETURNED shipments.
# 7. delivered_date cannot be earlier than shipped_date.
#
# RETURN_TO_ORIGIN is allowed as a status.
# It is not treated as a customer-delivered shipment.


validated_df = (
    standardized_df

    .withColumn(
        "_quarantine_reason",

        F.concat_ws(
            "; ",

            F.when(
                F.col("shipment_id").isNull()
                | (F.col("shipment_id") == ""),
                F.lit("Missing shipment_id")
            ),

            F.when(
                F.col("order_id").isNull()
                | (F.col("order_id") == ""),
                F.lit("Missing order_id")
            ),

            F.when(
                F.col("shipment_status").isNull()
                | (~F.col("shipment_status").isin(ALLOWED_STATUSES)),
                F.lit("Invalid shipment_status")
            ),

            F.when(
                F.col("shipped_date").isNull(),
                F.lit("Missing or invalid shipped_date")
            ),

            F.when(
                (F.col("shipment_status") == "DELIVERED")
                & F.col("delivered_date").isNull(),
                F.lit("Delivered shipment missing delivered_date")
            ),

            F.when(
                (~F.col("shipment_status").isin(
                    "DELIVERED",
                    "RETURNED"
                ))
                & F.col("delivered_date").isNotNull(),
                F.lit("Undelivered shipment has delivered_date")
            ),

            F.when(
                F.col("delivered_date").isNotNull()
                & F.col("shipped_date").isNotNull()
                & (
                    F.col("delivered_date")
                    < F.col("shipped_date")
                ),
                F.lit("delivered_date earlier than shipped_date")
            )
        )
    )
)

invalid_df = validated_df.filter(
    F.col("_quarantine_reason") != ""
)

valid_df = validated_df.filter(
    F.col("_quarantine_reason") == ""
)

valid_count = valid_df.count()
invalid_count = invalid_df.count()

print("VALID SHIPMENT RECORDS:", valid_count)
print("INVALID SHIPMENT RECORDS:", invalid_count)

display(
    invalid_df.select(
        "shipment_id",
        "order_id",
        "shipment_status",
        "shipped_date",
        "delivered_date",
        "_quarantine_reason"
    )
)


# 8. DUPLICATE SHIPMENT DETECTION

# Assumption: one current record per shipment_id.
# If the source represents shipment event history, this
# deduplication key must be changed.

dedup_window = (
    Window
    .partitionBy("shipment_id")
    .orderBy(
        F.col("_ingested_at").desc_nulls_last()
        if "_ingested_at" in valid_df.columns
        else F.col("shipment_id").asc(),

        F.col("_source_file").asc_nulls_last()
        if "_source_file" in valid_df.columns
        else F.col("shipment_id").asc(),

        F.col("tracking_number").asc_nulls_last()
    )
)

ranked_df = valid_df.withColumn(
    "_dedup_rank",
    F.row_number().over(dedup_window)
)

silver_candidates_df = ranked_df.filter(
    F.col("_dedup_rank") == 1
)

duplicate_df = (
    ranked_df
    .filter(F.col("_dedup_rank") > 1)
    .drop("_dedup_rank")
    .withColumn(
        "_quarantine_reason",
        F.lit("Duplicate shipment_id")
    )
)

duplicate_count = duplicate_df.count()

print("DUPLICATE RECORDS QUARANTINED:", duplicate_count)


# 9. FINAL SILVER DATAFRAME

silver_df = (
    silver_candidates_df
    .drop("_dedup_rank")
    .drop("_quarantine_reason")
)

silver_count = silver_df.count()

print("RECORDS AFTER DEDUPLICATION:", silver_count)


# 10. COMBINE INVALID AND DUPLICATE RECORDS

quarantine_df = (
    invalid_df
    .unionByName(
        duplicate_df,
        allowMissingColumns=True
    )
    .withColumn(
        "quarantined_at",
        F.current_timestamp()
    )
)

quarantine_count = quarantine_df.count()

print("TOTAL QUARANTINE RECORDS:", quarantine_count)

display(
    quarantine_df.select(
        "shipment_id",
        "order_id",
        "shipment_status",
        "_quarantine_reason"
    )
)


# 11. WRITE SILVER DELTA TABLE

(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print("Silver shipments table written successfully.")


# 12. WRITE QUARANTINE DELTA TABLE

(
    quarantine_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print("Shipment quarantine table written successfully.")


# 13. READ BACK AND VALIDATE SILVER TABLE

silver_result = spark.table(SILVER_TABLE)

final_silver_count = silver_result.count()

duplicate_groups = (
    silver_result
    .groupBy("shipment_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_silver_records = (
    silver_result
    .filter(
        F.col("shipment_id").isNull()
        | F.col("order_id").isNull()
        | F.col("shipment_status").isNull()
        | (~F.col("shipment_status").isin(ALLOWED_STATUSES))
        | F.col("shipped_date").isNull()
    )
    .count()
)

print("SILVER SHIPMENT RECORDS:", final_silver_count)
print("DUPLICATE SHIPMENT GROUPS:", duplicate_groups)
print("INVALID SILVER RECORDS:", invalid_silver_records)

if duplicate_groups == 0 and invalid_silver_records == 0:
    print("Silver shipment validation passed.")
else:
    raise Exception("Silver shipment validation failed.")


# 14. BRONZE-TO-SILVER RECONCILIATION

bronze_ids = (
    bronze_df
    .filter(
        F.col("shipment_id").isNotNull()
        & (F.trim(F.col("shipment_id").cast("string")) != "")
    )
    .select(
        F.trim(F.col("shipment_id").cast("string")).alias("shipment_id")
    )
    .distinct()
)

silver_ids = (
    silver_result
    .select("shipment_id")
    .distinct()
)

quarantine_ids = (
    quarantine_df
    .filter(F.col("shipment_id").isNotNull())
    .select("shipment_id")
    .distinct()
)

bronze_distinct_count = bronze_ids.count()
silver_distinct_count = silver_ids.count()

missing_ids_df = bronze_ids.join(
    silver_ids,
    on="shipment_id",
    how="left_anti"
)

missing_ids_count = missing_ids_df.count()

unexplained_missing_ids = (
    missing_ids_df
    .join(
        quarantine_ids,
        on="shipment_id",
        how="left_anti"
    )
    .count()
)

print("BRONZE DISTINCT SHIPMENT IDS:", bronze_distinct_count)
print("SILVER DISTINCT SHIPMENT IDS:", silver_distinct_count)
print("MISSING IDS FROM SILVER:", missing_ids_count)
print("MISSING IDS WITHOUT QUARANTINE EXPLANATION:",
      unexplained_missing_ids)

if unexplained_missing_ids == 0:
    print("Bronze-to-Silver reconciliation passed.")
else:
    display(
        missing_ids_df.join(
            quarantine_ids,
            on="shipment_id",
            how="left_anti"
        )
    )
    raise Exception("Bronze-to-Silver reconciliation failed.")


# 15. REFERENTIAL INTEGRITY - ORDER VALIDATION

ORDERS_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_orders"

orders_df = spark.table(ORDERS_TABLE)

shipment_order_refs = (
    silver_result
    .select("order_id")
    .distinct()
)

valid_order_refs = (
    orders_df
    .select("order_id")
    .distinct()
)

shipments_without_orders = (
    shipment_order_refs
    .join(
        valid_order_refs,
        on="order_id",
        how="left_anti"
    )
)

shipments_without_orders_count = shipments_without_orders.count()

print(
    "SHIPMENTS WITHOUT MATCHING SILVER ORDERS:",
    shipments_without_orders_count
)

display(shipments_without_orders.limit(30))


# 16. SHIPMENT STATUS SUMMARY

status_summary = (
    silver_result
    .groupBy("shipment_status")
    .agg(
        F.count("*").alias("shipment_count")
    )
    .orderBy(F.desc("shipment_count"))
)

display(status_summary)


# 17. CARRIER SUMMARY

carrier_summary = (
    silver_result
    .groupBy("carrier")
    .agg(
        F.count("*").alias("total_shipments"),

        F.sum(
            F.when(
                F.col("shipment_status") == "DELIVERED",
                1
            ).otherwise(0)
        ).alias("delivered_shipments"),

        F.sum(
            F.when(
                F.col("shipment_status") == "RETURN_TO_ORIGIN",
                1
            ).otherwise(0)
        ).alias("return_to_origin_shipments"),

        F.sum(
            F.when(
                F.col("shipment_status") == "RETURNED",
                1
            ).otherwise(0)
        ).alias("returned_shipments")
    )
    .orderBy(F.desc("total_shipments"))
)

display(carrier_summary)


# 18. DELIVERY DURATION ANALYSIS

delivery_duration_df = (
    silver_result
    .filter(
        (F.col("shipment_status") == "DELIVERED")
        & F.col("shipped_date").isNotNull()
        & F.col("delivered_date").isNotNull()
    )
    .withColumn(
        "delivery_duration_hours",
        (
            F.unix_timestamp("delivered_date")
            - F.unix_timestamp("shipped_date")
        ) / 3600
    )
)

delivery_duration_summary = (
    delivery_duration_df
    .agg(
        F.count("*").alias("delivered_shipments"),

        F.round(
            F.avg("delivery_duration_hours"),
            2
        ).alias("average_delivery_hours"),

        F.round(
            F.min("delivery_duration_hours"),
            2
        ).alias("minimum_delivery_hours"),

        F.round(
            F.max("delivery_duration_hours"),
            2
        ).alias("maximum_delivery_hours")
    )
)

display(delivery_duration_summary)


# 19. QUARANTINE REASON SUMMARY

final_quarantine_df = spark.table(QUARANTINE_TABLE)

quarantine_summary = (
    final_quarantine_df
    .groupBy("_quarantine_reason")
    .agg(
        F.count("*").alias("record_count")
    )
    .orderBy(F.desc("record_count"))
)

display(quarantine_summary)


# 20. FINAL PROJECT SUMMARY

final_bronze_count = spark.table(BRONZE_TABLE).count()
final_silver_count = spark.table(SILVER_TABLE).count()
final_quarantine_count = spark.table(QUARANTINE_TABLE).count()

print("========================================")
print("SILVER SHIPMENTS - FINAL SUMMARY")
print("========================================")

print("Bronze Records:", final_bronze_count)
print("Silver Records:", final_silver_count)
print("Invalid Records:", invalid_count)
print("Duplicate Records:", duplicate_count)
print("Total Quarantine Records:", final_quarantine_count)

print("Shipment IDs Without Silver Records:", missing_ids_count)
print("Unexplained Missing Shipment IDs:", unexplained_missing_ids)
print("Shipments Without Matching Orders:", shipments_without_orders_count)

print("========================================")

if (
    final_silver_count + final_quarantine_count
    == final_bronze_count
):
    print("ROW COUNT RECONCILIATION PASSED.")
else:
    print(
        "WARNING: Row count reconciliation differs. "
        "Check source rows with missing shipment_id."
    )

print("Silver Shipments notebook execution completed.")


In [0]:
quarantine_df = spark.table(
    "workspace.ecommerce_silver.quarantine_shipments"
)

display(
    quarantine_df.select(
        "shipment_id",
        "order_id",
        "shipment_status",
        "shipped_date",
        "delivered_date",
        "_quarantine_reason"
    )
)

In [0]:

display(
    quarantine_df
    .select(
        "shipment_id",
        "order_id",
        "shipment_status",
        "shipped_date",
        "delivered_date",
        "_quarantine_reason"
    )
    .limit(30)
)


In [0]:
ALLOWED_STATUSES = [
    "PENDING",
    "PROCESSING",
    "SHIPPED",
    "IN_TRANSIT",
    "OUT_FOR_DELIVERY",
    "DELIVERED",
    "CANCELLED",
    "RETURNED",
    "RETURN_TO_ORIGIN"
]

In [0]:
display(
    quarantine_df
    .filter(
        F.col("_quarantine_reason").contains(
            "Undelivered shipment has delivered_date"
        )
    )
    .select(
        "shipment_id",
        "order_id",
        "shipment_status",
        "shipped_date",
        "delivered_date",
        "_quarantine_reason"
    )
)